# One-Day VNP46A2 / VNP46A1 / VJ146A2 Stray-Light Comparison

This notebook compares the A1/VJ diagnostic outputs for the same review dates used in `stray_light_blackmarbler_redownload_comparison.ipynb`: `2023-10-20`, `2023-02-28`, and `2023-01-29`.

It does not download data and does not change production outputs.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATES = ["2023-10-20", "2023-02-28", "2023-01-29"]

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "STRAY_LIGHT_FIX_PLAN.md").exists() and (path / "blackmarbler").exists():
            return path
        nested = path / "6-codebases" / "repos" / "Reliability-Assessment"
        if (nested / "STRAY_LIGHT_FIX_PLAN.md").exists() and (nested / "blackmarbler").exists():
            return nested
    raise FileNotFoundError("Could not locate Reliability-Assessment repo root.")

ROOT = find_repo_root(Path.cwd())
OUT_DIR = ROOT / "blackmarbler" / "out_vnp46a2_sa_daily" / "qa_straylight_validation" / "one_day_a1_vj"
SEPARATE_PANEL_DIR = OUT_DIR / "png" / "separate_panels"

def paths_for_date(date: str) -> dict:
    return {
        "summary": OUT_DIR / f"one_day_a1_vj_summary_{date}.csv",
        "masks": OUT_DIR / f"one_day_a1_vj_mask_shares_{date}.csv",
        "panel": OUT_DIR / "png" / f"one_day_a1_vj_comparison_{date}.png",
    }

print(f"Repo root: {ROOT}")
print(f"Diagnostic outputs: {OUT_DIR}")
print(f"Dates: {', '.join(DATES)}")

In [ ]:
required = []
for date in DATES:
    p = paths_for_date(date)
    required.extend([p["summary"], p["masks"], p["panel"]])
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Run stray_light_one_day_a1_vj_diagnostic.R first. Missing:\n" + "\n".join(str(p) for p in missing))

summary = pd.concat([pd.read_csv(paths_for_date(date)["summary"]) for date in DATES], ignore_index=True)
masks = pd.concat([pd.read_csv(paths_for_date(date)["masks"]) for date in DATES], ignore_index=True)
summary["date"] = summary["date"].astype(str)
masks["date"] = masks["date"].astype(str)

summary

## Side-by-Side Map Panel

The panel uses a common radiance color scale across scenarios. The important comparison is whether the broad VNP46A2 contamination remains visible after each masking/replacement strategy.

In [ ]:
for date in DATES:
    img = plt.imread(paths_for_date(date)["panel"])
    fig, ax = plt.subplots(figsize=(16, 9.6))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"One-day A1/VJ diagnostic: {date}")
    plt.show()

## Separate Readable Panels

These are crops of the same diagnostic figure, shown one scenario at a time so the maps and legends are easier to inspect. The crops are also saved as standalone PNGs in `png/separate_panels/`.

In [ ]:
SEPARATE_PANEL_DIR.mkdir(parents=True, exist_ok=True)

# Fractions of the full generated panel image. Each crop includes its map title and legend.
panel_crops = {
    "Current VNP46A2": (0.00, 0.06, 0.34, 0.50),
    "VNP46A2 + A1 stray bit": (0.34, 0.06, 0.67, 0.50),
    "VNP46A2 + A1 strict DNB bits": (0.67, 0.06, 1.00, 0.50),
    "VJ146A2 candidate C2 mask": (0.00, 0.51, 0.34, 0.98),
    "Blend: VNP strict else VJ": (0.34, 0.51, 0.67, 0.98),
}

def crop_fraction(img, box):
    left, top, right, bottom = box
    return img.crop((int(left * w), int(top * h), int(right * w), int(bottom * h)))

for date in DATES:
    panel_img = Image.open(paths_for_date(date)["panel"])
    w, h = panel_img.size

    print(f"=== {date} ===")
    for scenario, box in panel_crops.items():
        cropped = crop_fraction(panel_img, box)
        out_png = SEPARATE_PANEL_DIR / (scenario.lower().replace(" ", "_").replace(":", "").replace("+", "plus").replace("/", "_") + f"_{date}.png")
        cropped.save(out_png)

        metrics = summary.loc[(summary["date"] == date) & (summary["scenario"] == scenario), ["valid_share", "p_lit", "lit_share_all"]]
        fig, ax = plt.subplots(figsize=(12, 7.5))
        ax.imshow(cropped)
        ax.axis("off")
        ax.set_title(f"{date} - {scenario}")
        plt.show()

        print(metrics.to_string(index=False))
        print(f"saved: {out_png}\n")

## Quantitative Comparison

`p_lit` is lit share among valid pixels. `lit_share_all` is lit share over all pixels in the South Africa-minus-Lesotho raster.

In [ ]:
display_cols = ["date", "scenario", "valid_share", "p_lit", "lit_share_all", "mean_rad_valid", "n_valid_pixels", "n_lit_pixels"]
summary[display_cols].sort_values(["date", "scenario"])

In [ ]:
comparisons = []
for date in DATES:
    date_rows = summary.loc[summary["date"] == date].copy()
    baseline = date_rows.loc[date_rows["scenario"] == "Current VNP46A2"].iloc[0]
    date_rows["delta_valid_share_vs_current"] = date_rows["valid_share"] - baseline["valid_share"]
    date_rows["delta_p_lit_vs_current"] = date_rows["p_lit"] - baseline["p_lit"]
    date_rows["delta_lit_share_all_vs_current"] = date_rows["lit_share_all"] - baseline["lit_share_all"]
    comparisons.append(date_rows)

comparison = pd.concat(comparisons, ignore_index=True)
comparison[["date", "scenario", "delta_valid_share_vs_current", "delta_p_lit_vs_current", "delta_lit_share_all_vs_current"]]

## A1 QF_DNB Mask Check

This confirms whether the VNP46A1 DNB QA bits actually reject any pixels on this date.

In [ ]:
masks

## Takeaway

For these bad-day examples, VNP46A1 `QF_DNB` masks should be checked date by date. `2023-10-20` shows that A1 masking does not reject the contaminated VNP46A2 swath, while VJ146A2 is much cleaner but lower coverage. The naive blend remains contaminated whenever VNP46A2 is not explicitly rejected.